# Capítulo 3 · Actividad: ¿qué hago con esta columna?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/04_actividad_decisiones_preprocesamiento.ipynb)

En las actividades anteriores te decíamos qué técnica aplicar. En la vida real nadie te lo dice: **tú decides**. En esta actividad vas a ver columnas de datasets que ya conoces y, para cada una, decidir qué es lo mejor: ¿one-hot u ordinal?, ¿logaritmo o no?, ¿estandarizar, min-max o nada?

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```

### Cómo trabajar

- En cada caso, **ejecuta la celda** que muestra los datos y **míralos con atención** antes de decidir.
- En las celdas ✍️ escribe **qué harías y por qué**. Una decisión sin justificación no vale.
- Algunos casos tienen una celda 🔍 **Comprueba** para verificar tu decisión con un par de líneas de código.
- Muchas veces no hay una única respuesta correcta, pero sí respuestas **mejor o peor justificadas**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"

# Datasets que ya conoces
propinas = pd.read_csv(DATA_DIR / "tips.csv")
pinguinos = pd.read_csv(DATA_DIR / "penguins.csv").dropna()
california = pd.read_csv(DATA_DIR / "california_housing.csv")

# Viviendas, con la limpieza de las clases anteriores
viviendas = pd.read_csv(DATA_DIR / "viviendas_sucio.csv").drop_duplicates().dropna()
viviendas["barrio"] = viviendas["barrio"].str.strip().str.capitalize()

---
## Parte 1 · ¿Cómo convierto esta columna en números?

### Caso 1 · La isla de los pingüinos

In [ ]:
pinguinos["island"].value_counts()

🤔 **Decide:** ¿cómo codificarías `island`: one-hot, ordinal o como binaria? ¿Por qué?

✍️ **Tu decisión:** usaría one-hot encoding porque island es una variable categórica nominal sin orden jerárquico natural (ninguna isla es mayor ni menor que otra). además solo son 3 categorías, por lo que genera pocas columnas y no sobrecarga el modelo. ordinal no conviene porque asumiría un orden que no existe y binaria no aplica porque son 3 islas.

### Caso 2 · El estado de las viviendas

In [ ]:
viviendas["estado"].value_counts()

🤔 **Decide:** ¿cómo codificarías `estado`? Si eliges ordinal, indica qué número le darías a cada valor.

✍️ **Tu decisión:** usaría codificación ordinal porque el estado de conservación sí tiene un orden de calidad natural y claro. le daría los números: 'a reformar': 0, 'buen estado': 1, 'como nuevo': 2. así reflejamos la progresión con una sola columna numérica.

🔍 **Comprueba:** codifica `estado` como decidiste y muestra cuántas viviendas hay de cada valor codificado.

In [ ]:
orden_estado = {'a reformar': 0, 'buen estado': 1, 'como nuevo': 2}
viviendas['estado_cod'] = viviendas['estado'].map(orden_estado)
viviendas['estado_cod'].value_counts().sort_index()

### Caso 3 · ¿Fumadores en la mesa?

In [ ]:
propinas["smoker"].value_counts()

🤔 **Decide:** ¿cuántas columnas necesitas para representar `smoker`? ¿Cómo la codificarías?

✍️ **Tu decisión:** solo necesitamos 1 columna. al ser una variable binaria con únicamente dos opciones ('yes' y 'no'), basta con codificarla como 0 y 1 (por ejemplo, 'no': 0 y 'yes': 1). crear dos columnas con one-hot sería redundante porque si no es fumador automáticamente sabemos que es no fumador.

### Caso 4 · Personas en la mesa

In [ ]:
propinas["size"].value_counts().sort_index()

🤔 **Decide:** `size` solo tiene 6 valores distintos. Un compañero propone tratarla como categórica y aplicarle one-hot. ¿Estás de acuerdo?

✍️ **Tu decisión:** no estoy de acuerdo. la variable size representa una cantidad real de personas por mesa, así que es numérica discreta. si le aplicamos one-hot perdemos la noción de escala y magnitud (que 4 comensales es el doble que 2) y generaríamos columnas innecesarias. lo mejor es dejarla como numérica tal cual.

### Caso 5 · El identificador

In [ ]:
print("Filas:", len(viviendas), "| Valores distintos de id:", viviendas["id"].nunique())
viviendas[["id", "barrio", "precio"]].head()

🤔 **Decide:** queremos predecir el precio. ¿Qué harías con la columna `id`?

✍️ **Tu decisión:** la eliminaría del dataset antes de entrenar el modelo. el id es solo un identificador aleatorio o correlativo para ordenar las filas y no aporta información sobre el precio de la casa. si la dejamos, el modelo podría memorizar identificadores específicos y sufrir sobreajuste (overfitting).

### Caso 6 · La fecha de venta

In [ ]:
print("Filas:", len(viviendas), "| Fechas distintas:", viviendas["fecha_venta"].nunique())
viviendas["fecha_venta"].head()

🤔 **Decide:** un compañero propone aplicar one-hot a `fecha_venta`. ¿Qué pasaría? ¿Qué harías tú con esta columna?

✍️ **Tu decisión:** si usamos one-hot generaríamos cientos de columnas casi vacías porque hay muchísimas fechas distintas (alta cardinalidad), provocando sobreajuste y lentitud. lo ideal es convertirla a datetime y extraer características numéricas útiles como el año, mes o día de la semana, o la antigüedad de la venta.

🔍 **Comprueba:** crea a partir de `fecha_venta` las variables que propusiste.

In [ ]:
fecha = pd.to_datetime(viviendas['fecha_venta'])
viviendas['año_venta'] = fecha.dt.year
viviendas['mes_venta'] = fecha.dt.month
viviendas[['fecha_venta', 'año_venta', 'mes_venta']].head()

### Caso 7 · El barrio y el modelo

In [ ]:
viviendas["barrio"].value_counts()

🤔 **Decide:** vas a codificar `barrio` con one-hot.

a) Si el modelo es una **regresión lineal**, ¿usarías `drop_first=True`? ¿Y si es un **árbol de decisión**?

b) El modelo se usará después en una web donde se publicarán viviendas **de barrios nuevos** que no están en estos datos. ¿Usarías `pd.get_dummies` o `OneHotEncoder`? ¿Con qué parámetro?

✍️ **Tu decisión:** a) en una regresión lineal sí usaría drop_first=True para evitar la trampa de las variables dummy (multicolinealidad perfecta entre columnas). en un árbol de decisión no hace falta drop_first=True porque los árboles evalúan variables por separado y no sufren problemas de colinealidad.

b) usaría OneHotEncoder de scikit-learn con el parámetro handle_unknown='ignore'. get_dummies no sabe cómo manejar categorías no vistas y rompería las columnas en producción, mientras que OneHotEncoder con ignore asigna ceros a todas las columnas dummy si llega un barrio nuevo desconocido.

---
## Parte 2 · ¿La transformo?

### Caso 8 · Ingresos de los distritos de California

In [ ]:
fig, ax = plt.subplots()
sns.histplot(california["MedInc"], bins=50, ax=ax)
ax.set_title(f"MedInc (asimetría = {california['MedInc'].skew():.2f})")
plt.show()

🤔 **Decide:** ¿le aplicarías un logaritmo a `MedInc`? ¿Por qué?

✍️ **Tu decisión:** sí le aplicaría logaritmo (o log1p) porque la distribución de ingresos tiene una asimetría positiva clara hacia la derecha (cola larga). la transformación logarítmica encoge esa cola, reduce la asimetría y hace que se parezca más a una distribución normal, lo que beneficia mucho a modelos como regresión lineal.

🔍 **Comprueba:** calcula la asimetría de `MedInc` después de aplicarle `np.log1p`.

In [ ]:
medinc_log = np.log1p(california['MedInc'])
print(f'asimetría original: {california["MedInc"].skew():.2f}')
print(f'asimetría con log1p: {medinc_log.skew():.2f}')

### Caso 9 · La aleta de los pingüinos

In [ ]:
fig, ax = plt.subplots()
sns.histplot(pinguinos["flipper_length_mm"], bins=30, ax=ax)
ax.set_title(f"flipper_length_mm (asimetría = {pinguinos['flipper_length_mm'].skew():.2f})")
plt.show()

🤔 **Decide:** el histograma no parece una campana: tiene dos "montañas". ¿Le aplicarías un logaritmo? ¿Por qué crees que tiene esa forma?

✍️ **Tu decisión:** no le aplicaría logaritmo. su asimetría ya es muy baja (0.35) y no tiene una cola larga; aplicar logaritmo no eliminaría las dos montañas. tiene esa forma bimodal porque el dataset combina varias especies de pingüinos con tamaños corporales muy diferentes (los pingüinos gentoo tienen aletas bastante más largas que los adélie y chinstrap).

### Caso 10 · Variables nuevas para California

Queremos predecir `MedHouseVal` (el valor de las viviendas). Un compañero propone crear estas tres variables:

| Variable | Fórmula |
|---|---|
| `habitaciones_por_persona` | `AveRooms / AveOccup` |
| `valor_por_habitacion` | `MedHouseVal / AveRooms` |
| `proporcion_dormitorios` | `AveBedrms / AveRooms` |

🤔 **Decide:** ¿cuáles de las tres se pueden usar en el modelo? ¿Por qué?

✍️ **Tu decisión:** solo se pueden usar habitaciones_por_persona y proporcion_dormitorios.

no se puede usar valor_por_habitacion porque utiliza MedHouseVal en la fórmula, que es justo la variable objetivo que queremos predecir. usarla causaría fuga de datos (data leakage) porque al poner el modelo en producción no conoceríamos el valor de la vivienda de antemano.

### Caso 11 · La antigüedad de las viviendas

In [ ]:
fig, ax = plt.subplots()
sns.histplot(california["HouseAge"], bins=52, ax=ax)
ax.set_title(f"HouseAge (asimetría = {california['HouseAge'].skew():.2f})")
plt.show()

🤔 **Decide:** ¿agruparías `HouseAge` en tramos ("nueva", "media", "antigua") o la dejarías numérica? ¿Le aplicarías un logaritmo?

✍️ **Tu decisión:** la dejaría como numérica continua. agruparla en tramos hace perder información cuantitativa precisa del año exacto. tampoco le aplicaría logaritmo porque su asimetría es prácticamente cero (0.06) y la distribución es casi uniforme. el pico al final simplemente se debe a que las edades superiores a 50 años se agruparon en 52.

---
## Parte 3 · ¿Escalo? ¿Cómo?

### Caso 12 · Clasificar pingüinos con KNN

Queremos predecir la especie con **KNN**, un modelo que clasifica cada pingüino según los pingüinos **más cercanos** (calcula distancias entre sus medidas).

In [ ]:
pinguinos[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]].describe().loc[["mean", "std", "min", "max"]].round(1)

🤔 **Decide:** ¿escalarías estas medidas? Si sí, ¿con qué método? ¿Qué pasaría si no las escalas?

✍️ **Tu decisión:** sí es indispensable escalarlas. usaría estandarización (StandardScaler) o min-max (MinMaxScaler). si no las escalamos, body_mass_g (que toma valores en miles de gramos) dominaría por completo el cálculo de distancias euclídeas, y variables con rangos pequeños en milímetros como bill_depth_mm no tendrían casi ninguna influencia en la predicción.

### Caso 13 · Los mismos pingüinos con un árbol de decisión

Ahora usamos un **árbol de decisión**, que clasifica haciendo preguntas del tipo *"¿la aleta mide más de 206 mm?"*.

🤔 **Decide:** ¿hace falta escalar las medidas? ¿Por qué?

✍️ **Tu decisión:** no hace falta escalar las medidas. los árboles de decisión eligen umbrales de corte considerando solo el orden relativo de los valores en cada variable por separado. como el escalado no altera el orden de los datos, las particiones y decisiones del árbol son idénticas con o sin escalado.

### Caso 14 · Imágenes para una red neuronal

Una red neuronal recibe imágenes en escala de grises. Cada píxel es un número de **0 (negro) a 255 (blanco)**:

In [ ]:
pixeles = np.random.default_rng(42).integers(0, 256, size=(4, 6))
pixeles

🤔 **Decide:** ¿qué escalado usarías: estandarización o min-max? ¿Por qué?

✍️ **Tu decisión:** usaría min-max (o simplemente dividir entre 255.0) para llevar los píxeles al rango [0, 1]. en imágenes conocemos con certeza los límites fijos teóricos (0 y 255) y no hay valores atípicos. min-max preserva la escala proporcional de intensidad de luz y facilita que el descenso de gradiente converja de forma estable.

### Caso 15 · ¿Escalo también las categóricas?

Tras codificar las viviendas, tienes columnas de este tipo:

In [ ]:
pd.concat([
    viviendas[["metros", "distancia_metro_m"]],
    pd.get_dummies(viviendas["barrio"], prefix="barrio", dtype=int),
    viviendas["garaje"].map({"no": 0, "sí": 1}).rename("garaje"),
], axis=1).head()

🤔 **Decide:** vas a estandarizar. ¿Lo aplicarías a **todas** las columnas, también a las de one-hot y a `garaje`? ¿Por qué?

✍️ **Tu decisión:** no estandarizaría las categóricas ni las columnas binarias como garaje. solo estandarizaría las variables continuas (metros y distancia_metro_m). las columnas con 0 y 1 ya están en un rango acotado que indica presencia o ausencia, y estandarizarlas destruiría su interpretabilidad directa sin mejorar el modelo.

### Caso 16 · ¿En qué orden?

Dos compañeros escriben este código para preparar los datos:

**Compañero A:**
```python
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.2, random_state=42)
```

**Compañero B:**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.fit_transform(X_test)
```

🤔 **Decide:** ¿alguno lo hace bien? Explica qué falla en cada uno y escribe (en palabras o en código) el orden correcto.

✍️ **Tu decisión:** ninguno lo hace bien.

- el compañero a comete fuga de datos (data leakage) porque ajusta el escalador con todo el dataset antes de separar train y test, filtrando información de test en el entrenamiento.
- el compañero b separa bien al inicio, pero vuelve a hacer fit_transform sobre test. en test nunca se hace fit, solo transform, para usar la media y varianza aprendidas en train.

el código correcto es:
```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)
```

---
## Parte 4 · Diseña el preprocesamiento completo

Queremos predecir la **propina** (`tip`) con una **regresión lineal**, a partir del resto de columnas de `propinas`.

In [ ]:
print(propinas.drop(columns=["tip"]).skew(numeric_only=True).round(2))
propinas.head()

🤔 **Decide:** completa la tabla con el tratamiento de cada columna y añade, si quieres, alguna variable nueva. Justifica las decisiones que no sean obvias.

| Columna | Tipo | Transformación | Codificación | Escalado |
|---|---|---|---|---|
| `total_bill` | Numérica continua | Logaritmo (`np.log1p`) | Ninguna | Estandarización (`StandardScaler`) |
| `sex` | Categórica binaria | Ninguna | Binaria 0/1 | Ninguno |
| `smoker` | Categórica binaria | Ninguna | Binaria 0/1 | Ninguno |
| `day` | Categórica nominal | Ninguna | One-Hot (`drop_first=True`) | Ninguno |
| `time` | Categórica binaria | Ninguna | Binaria 0/1 | Ninguno |
| `size` | Numérica discreta | Ninguna | Ninguna | Estandarización (`StandardScaler`) |
| `cuenta_por_persona` | Numérica continua | Ninguna | Ninguna | Estandarización (`StandardScaler`) |

✍️ **Tu decisión:** justificación de las decisiones:

- total_bill: tiene asimetría positiva clara (skew > 1), por lo que log1p compacta la cola larga y estabiliza la varianza. se estandariza porque la regresión lineal es sensible a la escala de los coeficientes.
- sex, smoker, time: son binarias, se codifican a 0 y 1 con una sola columna y no se escalan para conservar su significado directo de presencia/ausencia.
- day: es nominal de 4 días. usamos one-hot con drop_first=True para evitar la colinealidad en la regresión lineal.
- size: es numérica discreta y representa cantidad de personas; no usamos one-hot para mantener el orden y magnitud real. se estandariza junto con total_bill.
- variable nueva (cuenta_por_persona = total_bill / size): captura el consumo promedio por comensal, que ayuda a explicar la propina esperada sin causar fuga de datos porque no usa tip.